# 02b — Expression data: download, probe → gene mapping, common genes, QC
**Lymphoma AI Research · DLBCL prognosis**

Inputs: `data/clean/clinical_clean_all.csv`, `data/clean/split_GSE181063.csv` (from 02a).
Outputs: `data/processed/` (gene × sample matrices, gene table, QC table, manifest).

**Leakage rules for this notebook**
- Every data-driven choice for GSE181063 (log check, missing-probe rule, quantile reference, probe choice per gene, low-expression filter, QC thresholds) is computed on **training samples only** and then applied unchanged to test samples.
- No survival data is read or used in this notebook.
- External sets (GSE10846, GSE87371) are processed on their own, without outcome data.
- Variance filtering and z-scoring are **not** done here; they belong inside model building (step 4).

TR: Bu not defteri ifade verisini indirir, prob→gen eşler, ortak gen listesini çıkarır ve kalite kontrolü yapar. GSE181063 için bütün eşikler yalnızca eğitim örneklerinden hesaplanır; sağkalım verisine hiç dokunulmaz.

## 1. Setup and pre-specified parameters
TR: Drive'ı bağlar, klasörleri ve analizden önce sabitlenen parametreleri tanımlar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, gzip, io, json, hashlib, time, shutil, urllib.request, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata

PROJECT_DIR = '/content/drive/MyDrive/dlbcl-prognosis'
RAW      = f'{PROJECT_DIR}/data/raw'
EXPR_RAW = f'{RAW}/expression'
CLEAN    = f'{PROJECT_DIR}/data/clean'
PROC     = f'{PROJECT_DIR}/data/processed'
FIG      = f'{PROJECT_DIR}/figures/02b'
for d in (EXPR_RAW, PROC, FIG):
    os.makedirs(d, exist_ok=True)

SEED = 20261014
DATASETS = {'GSE181063': 'GPL14951', 'GSE10846': 'GPL570', 'GSE87371': 'GPL570'}

# Pre-specified parameters (fixed before looking at expression data)
PARAMS = {
    'log2_if_p99_above': 50,          # values above this at the 99th percentile -> treated as linear scale
    'max_probe_drop_frac': 0.10,      # stop if more than 10% of probes have missing values
    'low_expr_quantile': 0.25,        # drop genes in the bottom 25% of mean expression (GSE181063 train only)
    'probe_choice': 'highest_mean',   # one probe per gene: highest mean in reference samples
    'corr_outlier_robust_z': -4.0,    # flag samples whose median correlation is >4 robust SD below the reference
    'sex_y_genes': ['RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY', 'EIF1AY'],
    'sex_x_gene': 'XIST',
}
rng = np.random.default_rng(SEED)
print(json.dumps(PARAMS, indent=1))

## 2. Helper functions
TR: İndirme, SHA-256, series matrix okuma ve platform tablosu okuma fonksiyonları.

In [ ]:
def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def stub(acc):
    """GSE181063 -> GSE181nnn, GSE10846 -> GSE10nnn, GPL570 -> GPLnnn (GEO FTP folder naming)."""
    return acc[:-3] + 'nnn'

def list_ftp(url, pattern):
    with urllib.request.urlopen(url, timeout=120) as r:
        html = r.read().decode('utf-8', 'replace')
    return sorted(set(re.findall(pattern, html)))

def download(url, dest, retries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'  already on Drive: {os.path.basename(dest)} ({os.path.getsize(dest)/1e6:.0f} MB)')
        return dest
    tmp = dest + '.part'
    for k in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=600) as r, open(tmp, 'wb') as f:
                expected = int(r.headers.get('Content-Length', 0))
                shutil.copyfileobj(r, f, length=1 << 22)
            got = os.path.getsize(tmp)
            if expected and got != expected:
                raise IOError(f'incomplete download: {got} of {expected} bytes')
            os.replace(tmp, dest)
            print(f'  downloaded: {os.path.basename(dest)} ({got/1e6:.0f} MB)')
            return dest
        except Exception as e:
            print(f'  attempt {k+1} failed: {type(e).__name__}: {e}')
            time.sleep(10)
    raise RuntimeError(f'could not download {url}')

def matrix_urls(gse, gpl):
    base = f'https://ftp.ncbi.nlm.nih.gov/geo/series/{stub(gse)}/{gse}/matrix/'
    names = list_ftp(base, r'href="(GSE\d+(?:-GPL\d+)?_series_matrix\.txt\.gz)"')
    if any('-GPL' in n for n in names):          # multi-platform series: keep only our platform
        names = [n for n in names if f'-{gpl}_' in n]
    return [base + n for n in names]

def table_start(path):
    with gzip.open(path, 'rt', errors='replace') as f:
        for i, line in enumerate(f):
            if line.startswith('!series_matrix_table_begin'):
                return i + 1
    raise ValueError(f'no expression table in {path}')

def read_series_matrix(path, keep):
    """Read only the needed sample columns (saves RAM on large series)."""
    skip = table_start(path)
    header = pd.read_csv(path, sep='\t', skiprows=skip, nrows=0).columns
    cols = [header[0]] + [c for c in header[1:] if c in keep]
    df = pd.read_csv(path, sep='\t', skiprows=skip, usecols=cols, index_col=0, low_memory=False)
    df = df[~df.index.astype(str).str.startswith('!')]
    df.index = df.index.astype(str)
    return df.apply(pd.to_numeric, errors='coerce').astype('float32')

def read_platform_table(path):
    rows, inside = [], False
    with gzip.open(path, 'rt', errors='replace') as f:
        for line in f:
            if line.startswith('!') and line.rstrip().endswith('_table_begin'):
                inside = True; continue
            if line.startswith('!') and line.rstrip().endswith('_table_end'):
                break
            if inside:
                rows.append(line.rstrip('\n'))
    if not rows:
        raise ValueError(f'no platform table in {path}')
    return pd.read_csv(io.StringIO('\n'.join(rows)), sep='\t', dtype=str, low_memory=False)

print('helpers ready')

## 3. Load cleaned clinical IDs and the locked split
Only sample IDs, cohort and sex are used here (sex for the QC check). Survival columns are not touched.

TR: 2a'daki temiz hasta listelerini ve kilitli eğitim/test bölmesini okur. Sağkalım sütunlarına dokunmaz.

In [ ]:
clin = pd.read_csv(f'{CLEAN}/clinical_clean_all.csv', usecols=['sample_id', 'cohort', 'sex'])
split = pd.read_csv(f'{CLEAN}/split_GSE181063.csv')

ids = {g: clin.loc[clin['cohort'] == g, 'sample_id'].tolist() for g in DATASETS}
train_ids = split.loc[split['set'] == 'train', 'sample_id'].tolist()
test_ids  = split.loc[split['set'] == 'test',  'sample_id'].tolist()

assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(ids['GSE181063'])
for g in DATASETS:
    print(f'{g}: {len(ids[g])} samples')
print(f'GSE181063 train/test: {len(train_ids)}/{len(test_ids)}')

## 4. Download series matrices
Files go to `data/raw/expression/` on Drive and are reused if already there. The GSE181063 file can be large; this cell may take several minutes.

TR: GEO'dan ifade matrislerini Drive'a indirir (varsa tekrar indirmez) ve SHA-256 kaydeder. İlk çalıştırmada birkaç dakika sürebilir.

In [ ]:
raw_files = {}
for gse, gpl in DATASETS.items():
    urls = matrix_urls(gse, gpl)
    assert urls, f'{gse}: no series matrix found'
    print(f'{gse} ({gpl}): {len(urls)} file(s)')
    raw_files[gse] = [download(u, f'{EXPR_RAW}/{u.rsplit("/", 1)[-1]}') for u in urls]

raw_sha = {os.path.basename(p): sha256(p) for fs in raw_files.values() for p in fs}
for k, v in raw_sha.items():
    print(f'  {k}: {v[:12]}')

## 5. Read expression for the cleaned samples only
TR: Sadece 2a'da kalan hastaların sütunlarını okur; eksik örnek varsa durur.

In [ ]:
expr = {}
for gse, files in raw_files.items():
    parts = [read_series_matrix(p, set(ids[gse])) for p in files]
    df = pd.concat(parts, axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    missing = set(ids[gse]) - set(df.columns)
    assert not missing, f'{gse}: {len(missing)} cleaned samples not in matrix, e.g. {sorted(missing)[:3]}'
    expr[gse] = df[ids[gse]]
    print(f'{gse}: {df.shape[0]} probes x {len(ids[gse])} samples | NaN cells: {int(df.isna().sum().sum())}')

## 6. Scale check (log2) and missing values
The scale decision for GSE181063 uses training columns only. Probes with any missing value in the reference samples are dropped; leftover gaps in GSE181063 test samples are filled with the **training** median of that probe.

TR: Verinin log2 ölçekte olup olmadığını kontrol eder; eksik değerli probları atar. Test setindeki boşluklar eğitim medyanıyla doldurulur.

In [ ]:
ref_cols = {'GSE181063': train_ids, 'GSE10846': ids['GSE10846'], 'GSE87371': ids['GSE87371']}
scale_log = {}

for gse in DATASETS:
    df, ref = expr[gse], ref_cols[gse]
    v = df[ref].values
    p1, p50, p99 = np.nanpercentile(v, [1, 50, 99])
    if p99 > PARAMS['log2_if_p99_above']:
        df = np.log2(df.clip(lower=1.0))
        scale_log[gse] = 'linear -> log2(max(x,1))'
    else:
        scale_log[gse] = 'already log scale'
    print(f'{gse}: p1={p1:.2f} p50={p50:.2f} p99={p99:.2f} -> {scale_log[gse]}')

    bad = df[ref].isna().any(axis=1)
    frac = bad.mean()
    assert frac <= PARAMS['max_probe_drop_frac'], f'{gse}: {frac:.1%} probes with NaN, check the file'
    df = df.loc[~bad]
    n_fill = int(df.isna().sum().sum())
    if n_fill:
        med = df[ref].median(axis=1)
        df = df.T.fillna(med).T
    print(f'   dropped {int(bad.sum())} probes with NaN in reference; filled {n_fill} non-reference cells')
    expr[gse] = df.astype('float32')

## 7. Quantile normalisation to a reference distribution
Each sample is mapped **on its own** to a target distribution. For GSE181063 the target is built from training samples only, so test samples never influence it (this is the leakage-safe version of quantile normalisation). Each GPL570 set uses its own target.

TR: Her örneğin değer dağılımını bir referans dağılıma eşitler. GSE181063'te referans yalnızca eğitim örneklerinden yapılır; böylece test örnekleri eğitimi etkilemez.

In [ ]:
def quantile_to_reference(df, ref):
    target = np.sort(df[ref].values, axis=0).mean(axis=1)
    grid = np.arange(len(target))
    out = np.empty(df.shape, dtype='float32')
    for j in range(df.shape[1]):
        r = rankdata(df.iloc[:, j].values)          # average rank for ties, 1..n
        out[:, j] = np.interp(r - 1, grid, target)
    return pd.DataFrame(out, index=df.index, columns=df.columns)

before = {g: expr[g] for g in DATASETS}
norm = {g: quantile_to_reference(expr[g], ref_cols[g]) for g in DATASETS}

fig, axes = plt.subplots(3, 2, figsize=(10, 10))
for i, g in enumerate(DATASETS):
    show = list(rng.choice(ref_cols[g], size=min(30, len(ref_cols[g])), replace=False))
    for ax, d, t in ((axes[i, 0], before[g], 'before'), (axes[i, 1], norm[g], 'after')):
        ax.boxplot(d[show].values, showfliers=False)
        ax.set_title(f'{g} ({"train " if g == "GSE181063" else ""}30 random samples) - {t}', fontsize=9)
        ax.set_xticks([])
plt.tight_layout(); plt.savefig(f'{FIG}/boxplots_normalisation.png', dpi=120); plt.show()

## 8. Probe annotation (GPL570, GPL14951)
Uses NCBI's `.annot` files when available (same annotation source for both platforms → fewer symbol mismatches), otherwise the platform SOFT file. If both platforms have Entrez Gene IDs, genes are matched by Entrez ID (more stable than symbols); probes mapping to several genes (`///`) are removed.

TR: Prob kimliklerini genlere çevirmek için platform tablolarını indirir. Birden fazla gene giden problar atılır; mümkünse genler Entrez kimliğiyle eşleştirilir.

In [ ]:
def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    return next((low[c.lower()] for c in cands if c.lower() in low), None)

def get_annotation(gpl):
    base = f'https://ftp.ncbi.nlm.nih.gov/geo/platforms/{stub(gpl)}/{gpl}'
    for sub, fname in (('annot', f'{gpl}.annot.gz'), ('soft', f'{gpl}_family.soft.gz')):
        try:
            path = download(f'{base}/{sub}/{fname}', f'{EXPR_RAW}/{fname}')
            return read_platform_table(path), fname
        except Exception as e:
            print(f'  {gpl}: {fname} not usable ({type(e).__name__}: {e})')
    raise RuntimeError(f'no annotation for {gpl}')

def clean_map(tab):
    sym = pick(tab.columns, ['Gene symbol', 'Symbol', 'ILMN_Gene', 'GENE_SYMBOL'])
    eid = pick(tab.columns, ['Gene ID', 'Entrez_Gene_ID', 'ENTREZ_GENE_ID'])
    assert sym, f'no symbol column in {list(tab.columns)[:15]}'
    m = pd.DataFrame({'ID': tab['ID'].astype(str).str.strip(),
                      'symbol': tab[sym].astype(str).str.strip(),
                      'entrez': tab[eid].astype(str).str.strip().str.replace(r'\.0$', '', regex=True) if eid else np.nan})
    bad = {'', 'nan', 'NaN', 'None', '---'}
    m = m[~m['symbol'].isin(bad) & ~m['symbol'].str.contains('///')]
    if eid:
        m.loc[m['entrez'].isin(bad) | m['entrez'].str.contains('///', na=False), 'entrez'] = np.nan
    return m.drop_duplicates('ID'), sym, eid

annot, annot_src = {}, {}
for gpl in sorted(set(DATASETS.values())):
    tab, src = get_annotation(gpl)
    annot[gpl], sym, eid = clean_map(tab)
    annot_src[gpl] = {'file': src, 'symbol_col': sym, 'entrez_col': eid}
    cov = annot[gpl]['entrez'].notna().mean() if eid else 0
    print(f'{gpl}: {len(tab)} rows -> {len(annot[gpl])} single-gene probes | source {src} | Entrez coverage {cov:.1%}')

ok_entrez = all(annot_src[g]['entrez_col'] and annot[g]['entrez'].notna().mean() > 0.8 for g in annot)
GENE_KEY = 'entrez' if ok_entrez else 'symbol'
print('Gene key used for matching across platforms:', GENE_KEY)

for gse, gpl in DATASETS.items():
    rate = norm[gse].index.isin(annot[gpl]['ID']).mean()
    print(f'{gse}: {rate:.1%} of expression probes found in {gpl} annotation')
    assert rate > 0.5, f'{gse}: probe IDs do not match the annotation - stop and check'

## 9. Collapse probes to genes
One probe per gene: the probe with the highest mean expression in the reference samples (GSE181063: training only). This rule uses no outcome data.

TR: Aynı gene giden birden fazla prob varsa, referans örneklerde ortalaması en yüksek olanı seçer.

In [ ]:
def collapse(df, pmap, ref, key):
    m = pmap[pmap['ID'].isin(df.index)].dropna(subset=[key]).copy()
    m['ref_mean'] = df.loc[m['ID'], ref].mean(axis=1).values
    best = m.sort_values('ref_mean', ascending=False).drop_duplicates(key)
    out = df.loc[best['ID']].copy()
    out.index = best[key].values
    out.index.name = 'gene'
    return out, best[['ID', 'symbol', 'entrez', 'ref_mean']].set_index(best[key].values)

genes, probe_tab = {}, {}
for gse, gpl in DATASETS.items():
    genes[gse], probe_tab[gse] = collapse(norm[gse], annot[gpl], ref_cols[gse], GENE_KEY)
    print(f'{gse}: {norm[gse].shape[0]} probes -> {genes[gse].shape[0]} genes')

## 10. QC 1 — sex check (sample mix-up detection)
Expression-based sex is compared with clinical sex. Score = mean z of Y genes, minus XIST z **only if XIST is on the platform** (it is missing on GPL14951, so GSE181063 uses Y genes only).

Rule (fixed after the first run, using training/reference samples only, no outcome data):
- cut = midpoint between the median score of clinical males and clinical females in the reference samples
- ambiguous band = cut ± 25% of the gap between the two medians
- `sex_mismatch_confident` = clinical sex on the wrong side of the cut **and outside** the band → likely sample swap or clinical error
- `sex_mismatch_any` = wrong side of the cut, including the band (FFPE noise expected)

Samples are flagged, not removed; the exclusion rule goes into the step-4 analysis plan.

TR: Cinsiyeti Y genleri (ve varsa XIST) ile tahmin eder. GSE181063 platformunda XIST yok. İki küme arasındaki belirsiz bölgeye düşen örnekler "kesin uyuşmazlık" sayılmaz.

In [ ]:
sex_clin = clin.set_index('sample_id')['sex']
qc, sex_rule = [], {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, gse in zip(axes, DATASETS):
    df, pt, ref = genes[gse], probe_tab[gse], ref_cols[gse]
    s2k = {s: k for k, s in pt['symbol'].items()}
    yk = [s2k[g] for g in PARAMS['sex_y_genes'] if g in s2k]
    xk = s2k.get(PARAMS['sex_x_gene'])
    assert yk, f'{gse}: no Y genes found'
    z = df.sub(df[ref].mean(axis=1), axis=0).div(df[ref].std(axis=1), axis=0)
    yscore = z.loc[yk].mean()
    xscore = z.loc[xk] if xk is not None else pd.Series(np.nan, index=df.columns)
    score = yscore - xscore if xk is not None else yscore
    sc = sex_clin.loc[df.columns]
    m_med = float(score[ref][sc[ref] == 'M'].median()); f_med = float(score[ref][sc[ref] == 'F'].median())
    cut = (m_med + f_med) / 2; band = 0.25 * (m_med - f_med)
    pred = np.where(score > cut, 'M', 'F')
    d = pd.DataFrame({'sample_id': df.columns, 'cohort': gse, 'sex_clinical': sc.values,
                      'sex_predicted': pred, 'y_score': yscore.values, 'xist_score': xscore.values,
                      'sex_score': score.values})
    d['sex_mismatch_any'] = d['sex_clinical'].notna() & (d['sex_clinical'] != d['sex_predicted'])
    d['sex_mismatch_confident'] = d['sex_mismatch_any'] & ((d['sex_score'] - cut).abs() > band)
    sex_rule[gse] = {'y_genes': len(yk), 'xist': xk is not None, 'median_M': round(m_med, 3),
                     'median_F': round(f_med, 3), 'cut': round(cut, 3), 'band': round(band, 3)}
    qc.append(d)
    print(f'{gse}: {sex_rule[gse]} | any {int(d.sex_mismatch_any.sum())} | confident {int(d.sex_mismatch_confident.sum())}')
    for sx, col in (('M', 'tab:blue'), ('F', 'tab:red')):
        v = d.loc[d.sex_clinical == sx, 'sex_score']
        ax.hist(v, bins=40, alpha=0.5, color=col, label=f'clinical {sx}')
    ax.axvline(cut, c='k'); ax.axvspan(cut - band, cut + band, color='grey', alpha=0.2)
    ax.set_title(f'{gse} ({"Y only" if xk is None else "Y - XIST"})', fontsize=9)
    ax.set_xlabel('sex score'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIG}/sex_check.png', dpi=120); plt.show()
qc = pd.concat(qc, ignore_index=True)

## 11. Low-expression filter (GSE181063 training only) and the common gene list
Genes in the bottom 25% of mean expression in GSE181063 **training** samples are dropped. The common list is the intersection with genes measured in both GPL570 sets. No variance filter here (that is a modelling step).

TR: GSE181063 eğitim örneklerinde en düşük ifadeli %25 geni atar; kalanlarla iki GPL570 setinin kesişimini ortak gen listesi yapar.

In [ ]:
train_mean = genes['GSE181063'][train_ids].mean(axis=1)
thr = float(train_mean.quantile(PARAMS['low_expr_quantile']))
kept_181 = set(train_mean.index[train_mean > thr])
print(f'GSE181063: low-expression threshold (train) = {thr:.3f}; kept {len(kept_181)}/{len(train_mean)} genes')

g108, g873 = set(genes['GSE10846'].index), set(genes['GSE87371'].index)
common = sorted(kept_181 & g108 & g873)
print(f'GPL570 genes: GSE10846 {len(g108)}, GSE87371 {len(g873)}, both {len(g108 & g873)}')
print(f'Common gene list: {len(common)} genes')
lost = kept_181 - (g108 & g873)
print(f'GSE181063 genes kept but absent on GPL570: {len(lost)}')

gene_table = pd.DataFrame(index=pd.Index(common, name='gene'))
gene_table['symbol'] = probe_tab['GSE181063'].loc[common, 'symbol'].values
gene_table['symbol_GPL570'] = probe_tab['GSE10846'].loc[common, 'symbol'].values
for gse in DATASETS:
    gene_table[f'probe_{gse}'] = probe_tab[gse].loc[common, 'ID'].values
gene_table['train_mean_GSE181063'] = train_mean.loc[common].values
print('Symbol disagreements between platforms:', int((gene_table.symbol != gene_table.symbol_GPL570).sum()))

## 12. QC 2 — correlation outliers and PCA (training set)
For each sample: median correlation with the reference samples (self excluded). Robust z-score uses the reference median and MAD. GSE181063 test samples are scored against the training reference with the same rule. Flagged, not removed.

TR: Her örneğin diğer örneklerle medyan korelasyonunu hesaplar; diğerlerinden belirgin farklı olanları aykırı olarak işaretler. Eğitim setinde PCA grafiği çizer.

In [ ]:
def corr_outliers(df, ref):
    x = df.values
    x = (x - x.mean(axis=0)) / x.std(axis=0)
    c = (x.T @ x[:, [df.columns.get_loc(r) for r in ref]]) / x.shape[0]
    for j, r in enumerate(ref):
        c[df.columns.get_loc(r), j] = np.nan
    med = np.nanmedian(c, axis=1)
    is_ref = df.columns.isin(ref)
    center = np.median(med[is_ref]); mad = 1.4826 * np.median(np.abs(med[is_ref] - center))
    rz = (med - center) / mad
    return pd.DataFrame({'sample_id': df.columns, 'median_corr': med, 'corr_robust_z': rz,
                         'corr_outlier': rz < PARAMS['corr_outlier_robust_z']})

oc = pd.concat([corr_outliers(genes[g].loc[common], ref_cols[g]) for g in DATASETS], ignore_index=True)
qc = qc.merge(oc, on='sample_id')
qc['set'] = qc['sample_id'].map(dict(zip(split.sample_id, split.set))).fillna('external')
print(qc.groupby(['cohort', 'set'])[['corr_outlier', 'sex_mismatch_any', 'sex_mismatch_confident']].sum().astype(int))

# PCA on GSE181063 training samples only
X = genes['GSE181063'].loc[common, train_ids].T.values
X = (X - X.mean(0)) / X.std(0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
pcs = U[:, :2] * S[:2]; ve = S**2 / (S**2).sum()
flag = qc.set_index('sample_id').loc[train_ids, 'corr_outlier'].values
plt.figure(figsize=(6, 5))
plt.scatter(pcs[~flag, 0], pcs[~flag, 1], s=8, alpha=0.6, label='ok')
plt.scatter(pcs[flag, 0], pcs[flag, 1], s=30, c='red', label='corr outlier')
plt.xlabel(f'PC1 ({ve[0]:.1%})'); plt.ylabel(f'PC2 ({ve[1]:.1%})')
plt.title('GSE181063 training set, common genes'); plt.legend()
plt.tight_layout(); plt.savefig(f'{FIG}/pca_train.png', dpi=120); plt.show()

## 13. Save outputs and manifest
GSE181063 train and test are saved as **separate files** so the test set cannot be loaded by accident in steps 3–4.

TR: Matrisleri kaydeder. GSE181063 eğitim ve test ayrı dosyalara yazılır ki 3–4. adımlarda test seti yanlışlıkla açılmasın. Tüm dosyaların SHA-256 değerleri manifest'e yazılır.

In [ ]:
outputs = {
    'expr_GSE181063_train.parquet': genes['GSE181063'].loc[common, train_ids],
    'expr_GSE181063_test.parquet':  genes['GSE181063'].loc[common, test_ids],
    'expr_GSE10846.parquet':        genes['GSE10846'].loc[common, ids['GSE10846']],
    'expr_GSE87371.parquet':        genes['GSE87371'].loc[common, ids['GSE87371']],
}
for name, df in outputs.items():
    df.astype('float32').to_parquet(f'{PROC}/{name}')
    print(f'{name}: {df.shape[0]} genes x {df.shape[1]} samples')
gene_table.to_csv(f'{PROC}/gene_table_common.csv')
qc.to_csv(f'{PROC}/qc_samples.csv', index=False)

out_files = list(outputs) + ['gene_table_common.csv', 'qc_samples.csv']
manifest = {
    'notebook': '02b_expression_processing.ipynb',
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__,
    'seed': SEED, 'params': PARAMS, 'gene_key': GENE_KEY,
    'scale_decision': scale_log, 'annotation_sources': annot_src,
    'low_expr_threshold_train': thr, 'n_common_genes': len(common),
    'raw_sha256': raw_sha,
    'output_sha256': {f: sha256(f'{PROC}/{f}') for f in out_files},
    'sex_rule': sex_rule,
    'qc_counts': {k: {'corr_outlier': int(v.corr_outlier.sum()), 'sex_mismatch_any': int(v.sex_mismatch_any.sum()), 'sex_mismatch_confident': int(v.sex_mismatch_confident.sum())}
                  for k, v in qc.groupby(qc.cohort + '_' + qc.set)},
}
json.dump(manifest, open(f'{PROC}/manifest_02b.json', 'w'), indent=1)
print('saved manifest_02b.json')

## 14. Summary for the progress note
TR: İlerleme özeti için gereken sayıları tek yerde yazdırır. Bu çıktıyı bana yapıştır.

In [ ]:
print('Gene key:', GENE_KEY, '| annotation:', {g: v['file'] for g, v in annot_src.items()})
print('Scale:', scale_log)
for g in DATASETS:
    print(f'{g}: probes(after NaN drop)={norm[g].shape[0]} genes={genes[g].shape[0]}')
print(f'Low-expression threshold (train): {thr:.3f} | kept: {len(kept_181)} | common genes: {len(common)}')
print(qc.groupby(['cohort', 'set'])[['corr_outlier', 'sex_mismatch_any', 'sex_mismatch_confident']].sum().astype(int).to_string())
print('Sex rule:', sex_rule)
print('Raw SHA-256 (first 12):', {k: v[:12] for k, v in raw_sha.items()})